<a href="https://colab.research.google.com/github/DiegoB2002/BUS4-118S/blob/dev/PromptEngineeringExercise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2

In [ ]:
# ReACT-style iterative coding agent for Google Colab
# FIXED: uses subprocess instead of multiprocessing (avoids Colab worker_error)
# Features:
# - Clear ReACT stages with timestamps
# - Isolated execution with timeout
# - Captures stdout/stderr/traceback
# - Saves persistent logs to react_logs.txt

import sys
import os
import time
import traceback
import datetime
import textwrap
import subprocess
from dataclasses import dataclass
from typing import Optional, Dict, Any, Callable

# ============================================================
# LOGGER (live + persistent)
# ============================================================
class Logger:
    def __init__(self, logfile: str = "react_logs.txt", enable_file: bool = True):
        self.logfile = logfile
        self.enable_file = enable_file
        self._buf = []

    def _ts(self) -> str:
        return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def stage(self, name: str):
        sep = "=" * 80
        self.log(sep)
        self.log(name)
        self.log(sep)

    def log(self, msg: str, level: str = "INFO"):
        line = f"[{self._ts()}] [{level}] {msg}"
        self._buf.append(line)
        print(line, flush=True)

    def save(self):
        if not self.enable_file:
            return
        try:
            with open(self.logfile, "w", encoding="utf-8") as f:
                f.write("\n".join(self._buf))
            print(f"\nSaved logs to: {self.logfile}", flush=True)
        except Exception as e:
            print(f"\n[LOGGER ERROR] Could not save logs: {e}", flush=True)


# ============================================================
# SAFE RUNNER (subprocess + timeout + captured output)
# ============================================================
@dataclass
class RunResult:
    ok: bool
    exit_reason: str  # "success" | "exception" | "timeout" | "runner_error"
    stdout: str
    stderr: str
    tb: str  # traceback-like string (when available)


def run_with_timeout(code: str, timeout_s: float) -> RunResult:
    """
    Runs code in a fresh Python subprocess:
      python -c "<code>"
    This is the most reliable isolation mechanism in Colab/Jupyter.
    """
    try:
        # Note: pass code as argument (not through shell) to avoid quoting issues.
        completed = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=timeout_s
        )
        ok = (completed.returncode == 0)
        if ok:
            return RunResult(True, "success", completed.stdout, completed.stderr, "")
        else:
            # stderr typically contains the traceback from Python itself
            return RunResult(False, "exception", completed.stdout, completed.stderr, completed.stderr)
    except subprocess.TimeoutExpired as e:
        out = e.stdout if isinstance(e.stdout, str) else (e.stdout.decode("utf-8", "ignore") if e.stdout else "")
        err = e.stderr if isinstance(e.stderr, str) else (e.stderr.decode("utf-8", "ignore") if e.stderr else "")
        return RunResult(False, "timeout", out, err, f"Timeout after {timeout_s}s")
    except Exception:
        tb = traceback.format_exc()
        return RunResult(False, "runner_error", "", "", tb)


# ============================================================
# CODE GENERATOR (plug-in point)
# Replace this with an LLM-based generator later if desired.
# ============================================================
def default_code_generator(iteration: int, task_description: str, input_data: Any, last_result: Optional[RunResult]) -> str:
    attempts = [
        # Attempt 0: intentionally buggy (division by zero)
        """
divisor = 0
print("Attempt 0: dividing 10 by divisor...")
print(10 / divisor)
""",
        # Attempt 1: basic fix for divisor==0
        """
divisor = 0
print("Attempt 1: guarding against zero...")
if divisor == 0:
    print("divisor was 0; switching to 2")
    divisor = 2
print(10 / divisor)
""",
        # Attempt 2: robust handling across edge cases
        """
print("Attempt 2: robust safe_divide_10 with edge-case tests")

def safe_divide_10(divisor):
    if divisor is None:
        raise ValueError("divisor is None")
    try:
        d = float(divisor)
    except (TypeError, ValueError):
        raise ValueError(f"divisor must be numeric; got {divisor!r}")
    if d == 0.0:
        raise ZeroDivisionError("divisor is zero")
    return 10.0 / d

tests = [0, 2, "5", "oops", None, -4, 0.0, "0"]
for t in tests:
    try:
        print(f"divisor={t!r} -> {safe_divide_10(t)}")
    except Exception as e:
        print(f"divisor={t!r} -> ERROR: {type(e).__name__}: {e}")
"""
    ]
    idx = iteration if iteration < len(attempts) else len(attempts) - 1
    return textwrap.dedent(attempts[idx]).strip() + "\n"


# ============================================================
# ReACT AGENT
# ============================================================
class ReACTAgent:
    def __init__(
        self,
        task_description: str,
        input_data: Any = None,
        constraints: Optional[Dict[str, Any]] = None,
        code_generator: Callable[[int, str, Any, Optional[RunResult]], str] = default_code_generator,
        logfile: str = "react_logs.txt",
    ):
        self.task_description = task_description
        self.input_data = input_data
        self.constraints = constraints or {}
        self.max_iterations = int(self.constraints.get("max_iterations", 5))
        self.timeout_s = float(self.constraints.get("timeout_s", 5.0))
        self.code_generator = code_generator

        self.logger = Logger(logfile=logfile, enable_file=True)
        self.iteration = 0
        self.last_result: Optional[RunResult] = None

    # STAGE 1
    def reason_plan(self):
        self.logger.stage("STAGE 1 — REASON / PLAN")
        self.logger.log(f"Task: {self.task_description}")
        self.logger.log(f"Constraints: max_iterations={self.max_iterations}, timeout_s={self.timeout_s}")
        self.logger.log("Plan: generate code -> run (isolated subprocess) -> observe -> fix -> repeat")
        if self.input_data is not None:
            self.logger.log(f"Input data provided: type={type(self.input_data).__name__}")

    # STAGE 2
    def generate_code(self) -> str:
        self.logger.stage("STAGE 2 — GENERATE CODE")
        code = self.code_generator(self.iteration, self.task_description, self.input_data, self.last_result)
        preview = code if len(code) <= 1500 else (code[:1500] + "\n... (truncated)")
        self.logger.log(f"Generated code (iteration {self.iteration}):\n{preview}")
        return code

    # STAGE 3
    def run(self, code: str) -> RunResult:
        self.logger.stage("STAGE 3 — RUN")
        self.logger.log(f"Running isolated subprocess with timeout={self.timeout_s}s ...")
        res = run_with_timeout(code, timeout_s=self.timeout_s)
        self.logger.log(f"Run finished: exit_reason={res.exit_reason}", level=("INFO" if res.ok else "ERROR"))
        return res

    # STAGE 4
    def observe(self, res: RunResult):
        self.logger.stage("STAGE 4 — OBSERVE")
        self.logger.log("Captured STDOUT:")
        self.logger.log(res.stdout if res.stdout else "(empty)")
        self.logger.log("Captured STDERR:")
        self.logger.log(res.stderr if res.stderr else "(empty)")

        if not res.ok:
            if res.exit_reason == "timeout":
                self.logger.log("Diagnosis: code exceeded timeout; likely hung/too slow.", level="ERROR")
            elif res.exit_reason == "exception":
                self.logger.log("Diagnosis: code raised an exception (see STDERR/traceback).", level="ERROR")
            else:
                self.logger.log(f"Diagnosis: runner error: {res.exit_reason}", level="ERROR")

    # STAGE 5
    def fix(self, res: RunResult):
        self.logger.stage("STAGE 5 — FIX")
        if res.ok:
            self.logger.log("No fix needed (success).")
            return

        tb = res.tb or ""
        if res.exit_reason == "timeout":
            self.logger.log("Fix strategy: add termination conditions / reduce workload / raise timeout.", level="ERROR")
        elif "ZeroDivisionError" in tb:
            self.logger.log("Fix strategy: add divisor==0 checks / validation.", level="ERROR")
        elif "SyntaxError" in tb:
            self.logger.log("Fix strategy: correct syntax.", level="ERROR")
        elif "NameError" in tb:
            self.logger.log("Fix strategy: define missing names/imports.", level="ERROR")
        elif "FileNotFoundError" in tb:
            self.logger.log("Fix strategy: validate file paths before reading.", level="ERROR")
        elif "TypeError" in tb or "ValueError" in tb:
            self.logger.log("Fix strategy: improve input validation + conversions.", level="ERROR")
        else:
            self.logger.log("Fix strategy: general hardening; try next revision.", level="ERROR")

        self.iteration += 1
        self.logger.log(f"Incrementing iteration -> {self.iteration}")

    def solve(self) -> RunResult:
        self.reason_plan()

        final = RunResult(False, "no_runs", "", "", "")
        for _ in range(self.max_iterations):
            code = self.generate_code()
            res = self.run(code)
            self.last_result = res
            self.observe(res)

            if res.ok:
                self.logger.log("✅ Termination condition met: code executed successfully.")
                final = res
                break

            self.fix(res)
            final = res

        self.logger.log("Final summary:")
        self.logger.log(f"- ok={final.ok}")
        self.logger.log(f"- exit_reason={final.exit_reason}")
        self.logger.save()
        return final


# ============================================================
# DEMO RUN (edit task_description as needed)
# ============================================================
agent = ReACTAgent(
    task_description="Compute 10/divisor robustly; handle None, bad types, and zero; show outputs and logs.",
    input_data=None,
    constraints={"max_iterations": 5, "timeout_s": 5.0},
    logfile="react_logs.txt",
)

final_result = agent.solve()

print("\n" + "=" * 80)
print("COLAB OUTPUT SUMMARY")
print("=" * 80)
print("ok:", final_result.ok)
print("exit_reason:", final_result.exit_reason)
print("logfile: react_logs.txt (in the left file pane / working directory)")

[2026-03-05 04:15:36] [INFO] ================================================================================
[2026-03-05 04:15:36] [INFO] STAGE 1 — REASON / PLAN
[2026-03-05 04:15:36] [INFO] ================================================================================
[2026-03-05 04:15:36] [INFO] Task: Compute 10/divisor robustly; handle None, bad types, and zero; show outputs and logs.
[2026-03-05 04:15:36] [INFO] Constraints: max_iterations=5, timeout_s=5.0
[2026-03-05 04:15:36] [INFO] Plan: generate code -> run (isolated subprocess) -> observe -> fix -> repeat
[2026-03-05 04:15:36] [INFO] ================================================================================
[2026-03-05 04:15:36] [INFO] STAGE 2 — GENERATE CODE
[2026-03-05 04:15:36] [INFO] ================================================================================
[2026-03-05 04:15:36] [INFO] Generated code (iteration 0):
divisor = 0
print("Attempt 0: dividing 10 by divisor...")
print(10 / divisor)

[2026-03-05 04: